# YipitData Take-Home Assessment: End-to-End Pipeline & Hybrid Search

This notebook walks through the entire end-to-end flow of the project:
1. **Stage 1: Core ETL Pipeline**: Ingestion, cleaning, metadata enrichment, schema validation, and exporting the enriched dataset.
2. **Stage 2: Hybrid Semantic Search**: Generates sentence embeddings, computes top 3 similar articles, loads data into DuckDB, applies SQL filters, and exports the final canonical dataset.
3. **Stage 3: Interactive Search Demos**: Demonstrates how to use the `SemanticSearchEngine` programmatically for semantic and hybrid searches.

In [ ]:
import sys
from pathlib import Path

# Setup paths relative to the current workspace root
project_root = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
sys.path.extend([
    str(project_root),
    str(project_root / "search"),
    str(project_root / "pipeline")
])

import pandas as pd
import numpy as np
import duckdb
import pipeline
import search
from search_engine import SemanticSearchEngine

## Part 1: Run Core ETL Pipeline
We run the core ETL pipeline on the raw tech news CSV to output the enriched dataset.

In [ ]:
input_csv = project_root / "pipeline/data/input/tech_news.csv"
metadata_json = project_root / "pipeline/data/input/company_metadata.json"
enriched_csv = project_root / "pipeline/data/output/ai_articles_enriched.csv"

print("Running ETL pipeline...")
pipeline.run(
    input_path=input_csv,
    metadata_path=metadata_json,
    output_path=enriched_csv,
    apply_filter=False
)

# Display a preview of the enriched dataset
df_enriched = pd.read_csv(enriched_csv)
print(f"\nETL Pipeline complete. Loaded preview ({df_enriched.shape[0]} rows, {df_enriched.shape[1]} columns):")
df_enriched.head(3)

## Part 2: Run Hybrid Semantic Search Pipeline
Now we run the search pipeline, which will:
1. Read the enriched articles CSV.
2. Generate 384-dimensional sentence embeddings using the `all-MiniLM-L6-v2` model.
3. Compute the top 3 similar articles for each article.
4. Load everything into a DuckDB vector storage.
5. Query & export the filtered canonical schema to `search/output/filtered_ai_articles_with_embeddings.csv`.

In [ ]:
search_output_csv = project_root / "search/output/filtered_ai_articles_with_embeddings.csv"

print("Running Hybrid Semantic Search Pipeline...")
search.run_pipeline(
    input_path=enriched_csv,
    output_path=search_output_csv,
    model_name="all-MiniLM-L6-v2"
)

# Preview the exported filtered articles
df_filtered = pd.read_csv(search_output_csv)
print(f"\nSemantic search pipeline complete. Filtered size: {df_filtered.shape[0]} rows.")
df_filtered[["article_id", "title", "company_name", "category", "revenue_usd", "top_similar_articles"]].head(5)

## Part 3: Programmatic Usage & Interactive Search Demonstration
We initialize a `SemanticSearchEngine` and perform interactive semantic search and hybrid search queries.

In [ ]:
# Initialize the engine
engine = SemanticSearchEngine(model_name="all-MiniLM-L6-v2")

# Load the enriched articles
texts = (df_enriched["title"].fillna("") + ". " + df_enriched["summary"].fillna("")).tolist()
embeddings = engine.generate_embeddings(texts)

# Calculate top 3 most similar articles (excluding self) for each article
print("Calculating top 3 most similar articles for each row...")
similarity_matrix = np.dot(embeddings, embeddings.T)
norms = np.linalg.norm(embeddings, axis=1)
norms_matrix = np.outer(norms, norms)
similarity_matrix = similarity_matrix / np.maximum(norms_matrix, 1e-12)

top_similar_list = []
for idx in range(len(df_enriched)):
    sim_scores = similarity_matrix[idx]
    sorted_indices = np.argsort(sim_scores)[::-1]
    filtered_indices = [i for i in sorted_indices if i != idx]
    top_3_indices = filtered_indices[:3]
    top_3_ids = df_enriched.iloc[top_3_indices]["article_id"].tolist()
    top_similar_list.append(top_3_ids)

df_enriched["top_similar_articles"] = top_similar_list

# Load data into engine
engine.load_data(df_enriched, embeddings)

In [ ]:
# Vector similarity search for "NVIDIA financial results"
query = "NVIDIA financial results"
print(f"Executing Vector Similarity Search for: '{query}'")
results = engine.find_similar_articles(query, top_k=3)

# Display results in a readable format
results_df = []
for art_id, score in results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    results_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Summary": row["summary"]
    })
pd.DataFrame(results_df)

### Vector Similarity Search using an Article ID
We can also query using a specific `article_id`. The engine resolves the ID, retrieves its precomputed embedding, and finds the most similar articles in the corpus.

In [ ]:
# Query by article_id (e.g., "ART0003" which corresponds to NVIDIA profitability results)
query_id = "ART0003"
print(f"Executing Vector Similarity Search by Article ID: '{query_id}'")

# Find top 4 similar articles (the first result should be the article itself)
id_results = engine.find_similar_articles(query_id, top_k=4)

# Display results
id_results_df = []
for art_id, score in id_results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    id_results_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Summary": row["summary"]
    })
pd.DataFrame(id_results_df)

### Hybrid Search (SQL Filters + Vector Similarity)
We apply standard SQL filters first, then perform similarity ranking on the remaining records.

In [ ]:
# Hybrid Search: SQL Filters + Vector Similarity
query = "large language models"
sql_filter = "pub_year BETWEEN 2022 AND 2024 AND revenue_usd >= 50000000"

print(f"Executing Hybrid Search:")
print(f"  Query: '{query}'")
print(f"  SQL Filter: {sql_filter}")

hybrid_results = engine.hybrid_search(query, sql_filter, top_k=5)

# Display results
hybrid_df = []
for art_id, score in hybrid_results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    hybrid_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Revenue (USD)": f"${row['revenue_usd']:,}",
        "Year": row["pub_year"],
        "Summary": row["summary"]
    })
pd.DataFrame(hybrid_df)

### Direct Pipeline SQL Query Execution
We execute the exact pipeline SQL query (`test_pipeline_sql_query`) directly using DuckDB to retrieve the canonical schema.

In [ ]:
# Pipeline SQL query matching test_pipeline_sql_query
pipeline_query = """
SELECT 
    article_id,
    title,
    company_name,
    published_date,
    category,
    revenue_usd,
    summary,
    url,
    meta_industry AS industry,
    meta_founded_year AS founded_year,
    meta_headquarters AS headquarters,
    meta_employee_count AS employee_count,
    meta_is_public AS is_public,
    meta_stock_ticker AS stock_ticker,
    company_age,
    company_size_category,
    embedding,
    top_similar_articles
FROM articles
WHERE 
    (category = 'AI_ML' OR meta_industry = 'AI/ML')
    AND pub_year BETWEEN 2022 AND 2024
    AND revenue_usd >= 50000000
"""

print("Executing SQL query on DuckDB articles table...")
sql_df = engine.execute_sql(pipeline_query)

print(f"\nQuery executed successfully. Returned {sql_df.shape[0]} matching articles.")
sql_df[["article_id", "title", "company_name", "industry", "revenue_usd", "is_public"]].head(5)